<h3>Load in Dataset</h3>

In [ ]:
%matplotlib ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sunpy.map
from sunpy.net import Fido, attrs as a
from astropy import units as u
from astropy.coordinates import SkyCoord
import sunpy_soar
from glob import glob
from IPython.display import HTML
from astropy.io import fits
from astropy.time import Time
from datetime import datetime
#from pastamarkers import pasta, salsa
from stixpy.product import Product
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy

In [ ]:
plt.close('all')

In [ ]:
short_exposure_images = glob(r'C:\python_code\hrishort\*.fits')

In [ ]:
initial_map = sunpy.map.Map(short_exposure_images[216])

In [ ]:
tr_x=3200
tr_y=-550
bl_x=3000
bl_y=-800

In [ ]:
top_right = SkyCoord(tr_x * u.arcsec, tr_y * u.arcsec, frame=initial_map.coordinate_frame)
bottom_left = SkyCoord(bl_x * u.arcsec, bl_y * u.arcsec, frame=initial_map.coordinate_frame)
short_exposure_map_seq = initial_map.submap(bottom_left=bottom_left, top_right=top_right)
#short_exposure_map_seq.quicklook()

In [ ]:
second_map = sunpy.map.Map(short_exposure_images[217])
top_right = SkyCoord(tr_x * u.arcsec, tr_y * u.arcsec, frame=second_map.coordinate_frame)
bottom_left = SkyCoord(bl_x * u.arcsec, bl_y * u.arcsec, frame=second_map.coordinate_frame)
second_map_crop = second_map.submap(bottom_left=bottom_left, top_right=top_right)
short_exposure_map_seq = sunpy.map.Map([short_exposure_map_seq, second_map_crop], sequence = True)
third_map = sunpy.map.Map(short_exposure_images[218])
top_right = SkyCoord(tr_x * u.arcsec, tr_y * u.arcsec, frame=third_map.coordinate_frame)
bottom_left = SkyCoord(bl_x * u.arcsec, bl_y * u.arcsec, frame=third_map.coordinate_frame)
third_map_crop = third_map.submap(bottom_left=bottom_left, top_right=top_right)
short_exposure_map_seq = sunpy.map.Map(short_exposure_map_seq.maps + [third_map_crop], sequence = True)

In [ ]:
for i in range(65):
    next_map = sunpy.map.Map(short_exposure_images[217+i])
    top_right = SkyCoord(tr_x * u.arcsec, tr_y * u.arcsec, frame=next_map.coordinate_frame)
    bottom_left = SkyCoord(bl_x * u.arcsec, bl_y * u.arcsec, frame=next_map.coordinate_frame)
    next_map_crop = next_map.submap(bottom_left=bottom_left, top_right=top_right)
    short_exposure_map_seq = sunpy.map.Map(short_exposure_map_seq.maps + [next_map_crop], sequence = True)

In [ ]:
euitimes = []
for i in range(68):
    hdul = fits.open(short_exposure_images[216+i])
    hdu1 = hdul[1]
    euitimes.append(hdu1.header["DATE-BEG"])
    hdul.close()

In [ ]:
euitimes = Time(euitimes).to_datetime()

<h3>Mask dataset</h3>

In [ ]:
unmasked_map_seq = short_exposure_map_seq

In [ ]:
max = np.zeros(68)
for i in range(68):
    max[i] = (unmasked_map_seq[i]).max()

<h3>Apply threshold</h3>

In [ ]:
threshold = 0.03*max.max()
initial_masked_3 = short_exposure_map_seq[0]
xdim = int(initial_masked_3.dimensions[0].value)
ydim = int(initial_masked_3.dimensions[1].value)
for i in range(ydim):
    for j in range(xdim):
        if initial_masked_3.data[i,j] < threshold:
            initial_masked_3.data[i,j] = 0

In [ ]:
second_masked_3 = short_exposure_map_seq[1]
xdim = int(second_masked_3.dimensions[0].value)
ydim = int(second_masked_3.dimensions[1].value)
for i in range(ydim):
    for j in range(xdim):
        if second_masked_3.data[i,j] < threshold:
            second_masked_3.data[i,j] = 0

In [ ]:
masked_map_seq_3 = sunpy.map.Map([initial_masked_3, second_masked_3], sequence = True)

In [ ]:
for n in range(65):
    next_map = short_exposure_map_seq[2+n]
    xdim = int(next_map.dimensions[0].value)
    ydim = int(next_map.dimensions[1].value)
    for i in range(ydim):
        for j in range(xdim):
            if next_map.data[i,j] < threshold:
                next_map.data[i,j] = 0
    masked_map_seq_3 = sunpy.map.Map(masked_map_seq_3.maps + [next_map], sequence = True)

In [ ]:
max.max()

In [ ]:
lightcurves = np.zeros([471*558, len(masked_map_seq_3)])

In [ ]:
lightcurve_max = 0
for i in range(471):
    for j in range(558):
        for k in range(len(masked_map_seq_3)):
            #print(masked_map_seq_3[k].data[j,i])
            lightcurves[i*j, k] = masked_map_seq_3[k].data[j,i]
            if masked_map_seq_3[k].data[j,i] > lightcurve_max:
                lightcurve_max = masked_map_seq_3[k].data[j,i]

In [ ]:
initial_masked_3.peek()

In [ ]:
fig4 = plt.figure()#(figsize =(15, 15))
ax4 = fig4.add_subplot(111)
for n in range(471*558):
    #print(n)
    flag = False
    for k in range(len(masked_map_seq_3)):
        if lightcurves[n, k] != 0:
            flag = True
    if flag == True:
        try:
            ax4.plot(euitimes, (lightcurves[n,]), color = 'gray', alpha = 0.3)
        except:
            ax4.plot(euitimes[:67], (lightcurves[n,]), color = 'gray', alpha = 0.3)
#ax4.plot(euitimes, average_intensity_10, color = 'red')
ax4.set_xlabel("Time")
ax4.set_ylabel("Intensity")



#ax4.set_xlim(euitimes[10], euitimes[20])
plt.show()

<h2>Rise and Decay Times</h2>

| Threshold | FWHM rise time | Max Intensity | FWHM decay time | $t_{rise}$ | $t_{decay}$ |
| :- | :-: | :-: | :-: | :-: | :-: |
| $10\%$ | 57.78 | 60 | 62.52 | 2.22 | 2.52 |
| $20\%$ | 57.90 | 60 | 62.52 | 2.10 | 2.52 |
| $30\%$ | 57.90 | 60 | 62.64 | 2.10 | 2.64 |
| $50\%$ | 57.90 | 60 | 62.76 | 2.10 | 2.76 |